In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df_product=spark.read.format("parquet")\
    .load("abfss://bronze@dbenttoend.dfs.core.windows.net/products")

In [0]:
df_product=df_product.drop("_rescued_data")

In [0]:
df_product.display()

In [0]:
df_product.createOrReplaceTempView("products")

In [0]:
%sql
select * from products

In [0]:
%sql
create or replace function databricks_catalog.bronze.discount_func(p_price double)
returns double
language sql
return p_price * 0.90

In [0]:
%sql
select product_id, databricks_catalog.bronze.discount_func(price) as discounted_price from products

In [0]:
df_product=df_product.withColumn("discounted_price",expr("databricks_catalog.bronze.discount_func(price)"))

In [0]:
df_product.write.format("delta")\
  .mode("overwrite")\
  .save("abfss://silver@dbenttoend.dfs.core.windows.net/products")
  
 



In [0]:
%sql
create table if not exists databricks_catalog.silver.region_silver
using delta
location 'abfss://silver@dbenttoend.dfs.core.windows.net/region'